In [ ]:
import os
import math
import numpy as np
import pandas as pd
import scipy
import pydicom
from skimage import morphology, measure
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
import xgboost as xgb
import matplotlib.pyplot as plt
from tqdm import tqdm
import trimesh

# Configuration des chemins relatifs à ton arborescence
DATA_DIR = "../../dataset"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
CACHE_FILE = os.path.join(DATA_DIR, "radiomics_features_cache.csv")

# Chargement des métadonnées cliniques de base
df_train = pd.read_csv(TRAIN_CSV)
print(f"Métadonnées chargées : {df_train.shape[0]} lignes.")

In [ ]:
def load_and_sort_scan(patient_id, base_dir=TRAIN_IMG_DIR):
    patient_dir = os.path.join(base_dir, patient_id)
    filenames = os.listdir(patient_dir)

    slices = []
    for f in filenames:
        path = os.path.join(patient_dir, f)
        dicom_slice = pydicom.dcmread(path)

        # RGPD : Anonymisation des métadonnées sensibles à la volée
        # Dans un cadre de production, on remplacerait ou supprimerait ces tags
        dicom_slice.PatientName = "ANONYMOUS"
        dicom_slice.PatientID = patient_id

        slices.append(dicom_slice)

    # Tri spatial basé sur l'InstanceNumber (ou ImagePositionPatient si disponible)
    slices.sort(key=lambda x: float(x.InstanceNumber))

    return slices


# Test rapide sur le premier patient
example_patient = df_train["Patient"].iloc[0]
example_slices = load_and_sort_scan(example_patient)
print(f"Patient exemple : {example_patient}, Nombre de coupes : {len(example_slices)}")

In [ ]:
def get_pixels_hu(slices):
    image = np.stack([s.pixel_array for s in slices]).astype(np.int16)

    # Les pixels hors scanner sont souvent mis à une valeur très basse (-2000)
    image[image <= -1000] = -1000

    # Conversion en Unités Hounsfield (HU)
    for slice_idx in range(len(slices)):
        intercept = slices[slice_idx].RescaleIntercept
        slope = slices[slice_idx].RescaleSlope

        if slope != 1:
            image[slice_idx] = slope * image[slice_idx].astype(np.float64)
            image[slice_idx] = image[slice_idx].astype(np.int16)

        image[slice_idx] += np.int16(intercept)

    return np.array(image, dtype=np.int16)


def resample_volume(image, slices, new_spacing=[1, 1, 1]):
    # 1. Calcul du vrai espacement sur l'axe Z
    try:
        # On calcule la distance physique réelle entre la coupe 0 et la coupe 1
        z_spacing = np.abs(
            slices[0].ImagePositionPatient[2] - slices[1].ImagePositionPatient[2]
        )
    except AttributeError:
        # Fallback de secours si le tag est manquant
        z_spacing = slices[0].SliceThickness

    # Espacement [Z, X, Y]
    spacing = np.array(
        [z_spacing, slices[0].PixelSpacing[0], slices[0].PixelSpacing[1]],
        dtype=np.float32,
    )

    # 2. Calcul des facteurs de redimensionnement
    resize_factor = spacing / new_spacing
    new_real_shape = image.shape * resize_factor
    new_shape = np.round(new_real_shape)
    real_resize_factor = new_shape / image.shape

    # 3. Interpolation Spline pour combler les vides (mode 'nearest' pour aller vite,
    # mais un order=3 (bilinéaire) ou order=3 (cubique) est plus précis médicalement)
    resampled_image = scipy.ndimage.zoom(
        image, real_resize_factor, order=3, mode="nearest"
    )

    return resampled_image


# Relance ton test avec ça, ton volume devrait repasser autour de 3000-5000 cm³ !

# Application sur notre exemple
hu_volume = get_pixels_hu(example_slices)
resampled_volume = resample_volume(hu_volume, example_slices)

In [ ]:
def generate_lung_mask(volume_3d, plot_slice_index=None):
    z_slices, row_size, col_size = volume_3d.shape
    mask_3d = np.zeros_like(volume_3d, dtype=np.int8)

    # --- 1. CALCUL DU SEUIL GLOBAL ---
    mid_slice = volume_3d[z_slices // 2]
    mean_val, std_val = np.mean(mid_slice), np.std(mid_slice)

    if std_val > 0:
        img_norm = (mid_slice - mean_val) / std_val
        middle = img_norm[
            int(col_size / 5) : int(col_size / 5 * 4),
            int(row_size / 5) : int(row_size / 5 * 4),
        ]
        kmeans = KMeans(n_clusters=2, n_init=3, random_state=42).fit(
            np.reshape(middle, [-1, 1])
        )
        global_threshold = np.mean(kmeans.cluster_centers_)
    else:
        global_threshold = 0

    import scipy.ndimage

    # --- 2. APPLICATION DU SEUIL ET FILTRAGE ---
    for i in range(z_slices):
        img = volume_3d[i]

        c_mean, c_std = np.mean(img), np.std(img)
        if c_std == 0:
            continue
        img_norm = (img - c_mean) / c_std

        # --- MAGIE TOPOLOGIQUE : Détection de la frontière externe ---
        # 1. On identifie le padding artificiel (le coin 0,0 est toujours du padding ou de l'air pur)
        padding_val = img[0, 0]
        padding_mask = np.abs(img - padding_val) < 10
        fov_mask = ~padding_mask  # La vraie zone du scanner

        # 2. On remplit les trous (le patient) pour obtenir un cercle plein
        fov_mask = scipy.ndimage.binary_fill_holes(fov_mask)

        # 3. On extrait une bordure de 5 pixels à l'intérieur de ce cercle
        eroded_fov = morphology.erosion(fov_mask, np.ones([11, 11]))
        fov_boundary = fov_mask & ~eroded_fov
        # -------------------------------------------------------------

        # Binarisation pour trouver tout l'air (poumons + extérieur)
        thresh_img = np.where(img_norm < global_threshold, 1.0, 0.0)

        eroded = morphology.erosion(thresh_img, np.ones([3, 3]))
        eroded_dilation = morphology.dilation(eroded, np.ones([5, 5]))

        labels = measure.label(eroded_dilation)
        regions = measure.regionprops(labels)

        valid_regions = []

        for prop in regions:
            if prop.area < 200:
                continue

            region_mask = labels == prop.label

            # FILTRE TOPOLOGIQUE :
            # Si cette poche d'air touche la bordure du cercle du scanner,
            # c'est l'air ambiant ou le lit. Les poumons, eux, sont isolés à l'intérieur !
            if np.any(region_mask & fov_boundary):
                continue

            valid_regions.append(prop)

        # On trie par taille et on garde les 2 plus grandes poches internes (les poumons)
        valid_regions.sort(key=lambda x: x.area, reverse=True)
        final_labels = [r.label for r in valid_regions[:2]]

        slice_mask = np.zeros([row_size, col_size], dtype=np.int8)
        for N in final_labels:
            slice_mask += labels == N

        # Dilatation finale
        final_mask = morphology.dilation(slice_mask, np.ones([8, 8]))
        mask_3d[i] = final_mask

        # --- 3. AFFICHAGE DES ÉTAPES ---
        if plot_slice_index is not None and i == plot_slice_index:
            steps = [
                img,
                fov_boundary,
                eroded_dilation,
                slice_mask,
                final_mask,
                img * final_mask,
            ]
            titles = [
                "1. Originale",
                "2. Frontière Externe",
                "3. Air Brut",
                "4. Poumons Internes",
                "5. Masque Final",
                "6. Résultat",
            ]
            cmaps = ["bone", "Reds", "gray", "gray", "gray", "bone"]

            fig, axes = plt.subplots(2, 3, figsize=(20, 10))
            for idx, ax in enumerate(axes.flatten()):
                if idx < len(steps):
                    ax.imshow(steps[idx], cmap=cmaps[idx])
                    ax.set_title(titles[idx], fontsize=14, pad=10)
                ax.axis("off")
            plt.tight_layout()
            plt.show()

    # --- 4. FILTRAGE 3D POUR RETIRER LES ARTEFACTS ---
    # On labellise les composantes connexes en 3D
    labels_3d = measure.label(mask_3d)
    regions_3d = measure.regionprops(labels_3d)

    if len(regions_3d) > 0:
        # On trie par volume (aire en 3D)
        regions_3d.sort(key=lambda x: x.area, reverse=True)
        max_volume = regions_3d[0].area

        # On garde uniquement les composantes dont le volume représente au moins 5% du volume maximal
        valid_labels_3d = [r.label for r in regions_3d if r.area > max_volume * 0.05]

        mask_3d_clean = np.zeros_like(mask_3d, dtype=np.int8)
        for label in valid_labels_3d:
            mask_3d_clean += labels_3d == label

        mask_3d = mask_3d_clean

    return mask_3d.astype(bool)


# Lancement sur la coupe du milieu :
mask_3d = generate_lung_mask(
    resampled_volume, plot_slice_index=resampled_volume.shape[0] // 2
)
print(f"Masque généré. Pixels dans le masque : {np.sum(mask_3d)}")

In [ ]:
# def plot_3d_lung_mesh(mask_3d, threshold=0.5):
#     # L'algorithme Marching Cubes génère les sommets et les faces
#     # mask_3d original = (Z, Y, X). Transposé = (Y, X, Z)
#     mask_3d_transposed = np.transpose(mask_3d, (1, 2, 0))
#     verts, faces, normals, values = measure.marching_cubes(
#         mask_3d_transposed, level=threshold
#     )

#     fig = plt.figure(figsize=(10, 10))
#     ax = fig.add_subplot(111, projection="3d")

#     # Création de l'objet 3D
#     mesh = Poly3DCollection(verts[faces], alpha=0.3)
#     face_color = [0.45, 0.45, 0.75]  # Bleu grisé façon radiographie
#     mesh.set_facecolor(face_color)
#     ax.add_collection3d(mesh)

#     # --- LES CORRECTIONS SONT ICI ---
#     # 1. On ajuste les limites sur les dimensions transposées (Y, X, Z)
#     ax.set_xlim(0, mask_3d.shape[1])  # Axe X du plot = Lignes de l'image
#     ax.set_ylim(0, mask_3d.shape[2])  # Axe Y du plot = Colonnes de l'image
#     ax.set_zlim(0, mask_3d.shape[0])  # Axe Z du plot = Coupes (Slices)

#     # 2. On inverse l'axe Z pour remettre la tête en haut (Slice 0 en haut)
#     ax.invert_zaxis()

#     # 3. (Optionnel) Inverser l'axe Y car l'origine d'une image (0,0) est en haut à gauche
#     ax.invert_yaxis()

#     ax.set_xlabel("Axe X (Lignes)")
#     ax.set_ylabel("Axe Y (Colonnes)")
#     ax.set_zlabel("Axe Z (Coupes)")
#     ax.set_title("Reconstruction 3D du Masque Pulmonaire")

#     # 4. On corrige l'aspect ratio pour qu'il suive le bon ordre (Y, X, Z)
#     ax.set_box_aspect((mask_3d.shape[1], mask_3d.shape[2], mask_3d.shape[0]))

#     plt.show()


# # ATTENTION : Le rendu 3D avec matplotlib peut prendre quelques secondes
# plot_3d_lung_mesh(mask_3d)

In [ ]:
def create_3d_file(mask_3d, threshold=0.5, output_path="lung_mask.glb"):

    mask_3d_transposed = np.transpose(mask_3d, (1, 2, 0))

    verts, faces, normals, values = measure.marching_cubes(
        mask_3d_transposed, level=threshold
    )

    verts[:, 2] = mask_3d.shape[0] - verts[:, 2]

    verts[:, 1] = mask_3d.shape[2] - verts[:, 1]

    transform = trimesh.transformations.rotation_matrix(np.radians(-90), [1, 0, 0])

    mesh = trimesh.Trimesh(
        vertices=verts, faces=faces, vertex_normals=normals, process=False
    )

    mesh.apply_transform(transform)

    mesh.export(output_path)

    print(f"Fichier créé : {output_path}")


create_3d_file(mask_3d, output_path="lung_example_patient.glb")

In [ ]:
def plot_stack(
    volume_3d, start_with=0, cols=6, show_max=30, cmap="gray", title="Slice"
):
    """
    Affiche une grille des coupes 2D extraites d'un volume 3D (Z, Y, X)
    avec une distance fixe entre chaque image et un respect du ratio d'image.
    """
    total_slices = volume_3d.shape[0]
    available_slices = total_slices - start_with

    if available_slices <= 0:
        print("Aucune coupe à afficher avec ces paramètres.")
        return

    step = max(1, math.ceil(available_slices / show_max))
    indices_to_show = list(range(start_with, total_slices, step))
    num_images = len(indices_to_show)

    rows = max(1, math.ceil(num_images / cols))

    # --- NOUVEAUTÉ : Calcul dynamique des dimensions ---
    # Récupération des dimensions d'une coupe (Hauteur, Largeur)
    img_h, img_w = volume_3d.shape[1], volume_3d.shape[2]

    # 1. On fixe la largeur d'une sous-figure (ex: 4 pouces)
    cell_width = 4.0

    # 2. La hauteur de la sous-figure s'adapte au ratio de l'image (pour ne rien déformer)
    cell_height = cell_width * (img_h / img_w)

    # 3. Espacement entre les sous-figures (fraction de la largeur/hauteur d'une cellule)
    # w_space = 0.05 signifie un espace horizontal équivalent à 5% de la largeur de la cellule
    w_space_fraction = 0.05
    h_space_fraction = (
        0.1  # Un peu plus d'espace en hauteur pour laisser place au titre
    )

    # 4. Calcul de la taille globale de la figure (figsize)
    # On multiplie par (1 + fraction_espace) pour anticiper la place prise par les "trous"
    total_fig_width = cols * cell_width * (1 + w_space_fraction)
    total_fig_height = rows * cell_height * (1 + h_space_fraction)

    # Création de la figure avec un gridspec pour forcer les espacements
    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=[total_fig_width, total_fig_height],
        gridspec_kw={"wspace": w_space_fraction, "hspace": h_space_fraction},
    )

    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < num_images:
            real_idx = indices_to_show[i]
            ax.imshow(volume_3d[real_idx], cmap=cmap)
            ax.set_title(f"{title} {real_idx}")

        ax.axis("off")

    # ATTENTION : Ne plus utiliser plt.tight_layout() ici,
    # car il chercherait à écraser nos espacements (wspace, hspace).

    plt.show()


# 1. Afficher l'évolution du scan original (Max 30 coupes)
print("--- SCANNERS ORIGINAUX (HU) ---")
plot_stack(resampled_volume, start_with=0, show_max=32, cmap="bone", title="HU Z=")

# 2. Afficher le mask obtenu
print("--- SCANNERS ORIGINAUX (HU) ---")
plot_stack(mask_3d, start_with=0, show_max=32, cmap="bone", title="HU Z=")

# 3. Afficher le résultat final (Scanners segmentés)
# On multiplie le volume par le masque pour effacer l'extérieur
segmented_volume = resampled_volume * mask_3d
print("--- POUMONS ISOLES ---")
plot_stack(segmented_volume, start_with=0, show_max=32, cmap="bone", title="Seg Z=")

In [ ]:
def extract_radiomics_features(volume_3d, mask_3d):

    lung_pixels = volume_3d[mask_3d]

    if len(lung_pixels) == 0:
        return [0, 0, 0, 0]

    # Volume total (chaque voxel rééchantillonné fait 1mm3 = 0.001 cm3)
    volume_cm3 = round(len(lung_pixels) * 0.001, 3)

    # Statistiques de premier ordre de la distribution HU
    mean_hu = np.mean(lung_pixels)
    std_hu = np.std(lung_pixels)

    # Score de fibrose (ratio de tissus denses > -250 HU à l'intérieur du poumon)
    fibrosis_ratio = np.sum(lung_pixels > -250) / len(lung_pixels)

    return [volume_cm3, mean_hu, std_hu, fibrosis_ratio]


features_example = extract_radiomics_features(resampled_volume, mask_3d)
print(
    f"Caractéristiques extraites [Volume (cm3), Moyenne HU, Std HU, Ratio Fibrose] :\n{features_example}"
)

In [ ]:
# On veut traiter TOUS les patients de l'entraînement
unique_patients = df_train["Patient"].unique()
colonnes_features = ["Vol_Lung_cm3", "Mean_HU", "Std_HU", "Fibrosis_Ratio"]

# 1. Vérification du cache et identification des patients déjà traités
if os.path.exists(CACHE_FILE):
    print(f"Cache existant trouvé : {CACHE_FILE}")
    df_features = pd.read_csv(CACHE_FILE)
    patients_traites = df_features["Patient"].unique()
else:
    print("Aucun cache trouvé. Initialisation du fichier...")
    # Création d'un fichier CSV vide avec uniquement les en-têtes
    df_vide = pd.DataFrame(columns=["Patient"] + colonnes_features)
    df_vide.to_csv(CACHE_FILE, index=False)
    patients_traites = []
    df_features = df_vide

# 2. Filtrer pour ne garder que les patients non traités
patients_a_traiter = [p for p in unique_patients if p not in patients_traites]

if len(patients_a_traiter) == 0:
    print("Tous les patients ont déjà été traités ! Chargement direct.")
else:
    print(
        f"Extraction des features pour {len(patients_a_traiter)} patients restants..."
    )

    # Utilisation de tqdm pour avoir une barre de progression
    for p_id in tqdm(patients_a_traiter, desc="Radiomique"):
        try:
            slices = load_and_sort_scan(p_id)
            vol = get_pixels_hu(slices)
            vol_res = resample_volume(vol, slices)
            mask = generate_lung_mask(vol_res)

            # Extraction (renvoie normalement une liste/tuple de 4 valeurs)
            features = extract_radiomics_features(vol_res, mask)

            # Création d'un DataFrame d'une seule ligne pour ce patient
            row_df = pd.DataFrame([features], columns=colonnes_features)
            row_df.insert(0, "Patient", p_id)  # Ajout de la colonne Patient au début

            # 3. Sauvegarde immédiate dans le CSV en mode "append" (ajout)
            row_df.to_csv(CACHE_FILE, mode="a", header=False, index=False)

        except Exception as e:
            print(f"Erreur inattendue sur le patient {p_id}: {e}")

    # 4. Rechargement propre du dataset complet à la fin de la boucle
    df_features = pd.read_csv(CACHE_FILE)
    print("Mise en cache terminée avec succès !")

# Filtrage : On exclut les patients dont toutes les métriques de poumon sont à 0 ([0, 0, 0, 0])
df_features = df_features[~(df_features[colonnes_features] == 0).all(axis=1)]

# Fusion avec les métadonnées cliniques
df_ml = pd.merge(df_train, df_features, on="Patient", how="inner")
print(f"Dataset prêt pour le ML : {df_ml.shape[0]} lignes.")

In [ ]:
# Encodage des variables catégorielles
df_ml = pd.get_dummies(
    df_ml, columns=["Sex", "SmokingStatus"], drop_first=True, dtype=int
)

# Création du jeu de données d'entraînement historique (Patient-Semaine)
# Pour chaque ligne (semaine cible), on veut injecter la FVC de la première semaine connue (semaine de base)
base_df = (
    df_ml.sort_values(["Patient", "Weeks"]).groupby("Patient").first().reset_index()
)
base_df = base_df[["Patient", "Weeks", "FVC", "Percent"]].rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)

df_final = pd.merge(df_ml, base_df, on="Patient", how="inner")
df_final["Weeks_Delta"] = df_final["Weeks"] - df_final["Base_Week"]

# Suppression des colonnes non prédictives
X = df_final.drop(columns=["Patient", "FVC", "Percent"])
y = df_final["FVC"]

In [ ]:
# Définition de la métrique Laplace Log Likelihood pour l'évaluation
def laplace_log_likelihood(y_true, y_pred, sigma):
    sigma_clipped = np.maximum(sigma, 70)
    delta = np.minimum(np.abs(y_true - y_pred), 1000)
    metric = -(np.sqrt(2) * delta / sigma_clipped) - np.log(np.sqrt(2) * sigma_clipped)
    return np.mean(metric)


# ============================================================
# QUANTILE REGRESSION ML — ESTIMATION D'INCERTITUDE PAR ML
# ============================================================
# On entraîne 5 modèles XGBoost indépendants, un par quantile.
# L'objectif 'reg:quantileerror' optimise directement la pinball loss.
# Cela permet d'obtenir des intervalles de confiance asymétriques
# basés sur les données, sans formule mathématique fixe.

QUANTILES = [0.025, 0.10, 0.50, 0.90, 0.975]  # IC 95% et IC 80%

# Modèle central (prédiction ponctuelle de la FVC)
model_fvc = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

# Un modèle XGBoost par quantile
quantile_models = {
    q: xgb.XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=q,
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )
    for q in QUANTILES
}

# Stratification sur le sexe (variable catégorielle bien distribuée)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_laplace = []
coverages_90 = []  # Taux de couverture de l'intervalle [Q05, Q95]
coverages_60 = []  # Taux de couverture de l'intervalle [Q20, Q80]

print("Début de la validation croisée (Quantile Regression ML)...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, df_final["Sex_Male"])):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # 1. Entraînement du modèle central FVC
    model_fvc.fit(X_train, y_train)
    preds_fvc_val = model_fvc.predict(X_val)

    # 2. Entraînement et prédiction de chaque modèle quantile
    q_preds_val = {}
    for q, qmodel in quantile_models.items():
        qmodel.fit(X_train, y_train)
        q_preds_val[q] = qmodel.predict(X_val)

    # 3. Sigma ML = demi-largeur de l'intervalle [Q05, Q95]
    #    On clip à 70 minimum pour la métrique Laplace
    sigma_ml = np.maximum((q_preds_val[0.975] - q_preds_val[0.025]) / 2.0, 70)  # IC 95%

    # 4. Métriques d'évaluation
    score = laplace_log_likelihood(y_val.values, preds_fvc_val, sigma_ml)
    scores_laplace.append(score)

    # Coverage : pourcentage de vraies valeurs dans l'intervalle prédit
    in_90 = np.mean(
        (y_val.values >= q_preds_val[0.025]) & (y_val.values <= q_preds_val[0.975])
    )  # IC 95%
    in_60 = np.mean(
        (y_val.values >= q_preds_val[0.10]) & (y_val.values <= q_preds_val[0.90])
    )  # IC 80%
    coverages_90.append(in_90)
    coverages_60.append(in_60)

    print(
        f"Fold {fold + 1} | Laplace LLH: {score:.4f} | Coverage IC95% [Q2.5,Q97.5]: {in_90 * 100:.1f}% | Coverage IC80% [Q10,Q90]: {in_60 * 100:.1f}%"
    )

print(f"\n---> Laplace LLH Moyen       : {np.mean(scores_laplace):.4f}")
print(
    f"---> Coverage IC 95% [Q2.5,Q97.5] Moyen : {np.mean(coverages_90) * 100:.1f}% (cible >= 95%)"
)
print(
    f"---> Coverage IC 80% [Q10,Q90] Moyen : {np.mean(coverages_60) * 100:.1f}% (cible >= 80%)"
)
print("\nNote: Un bon modèle quantile doit avoir Coverage IC95% >= 95% et IC80% >= 80%")

In [ ]:
# ============================================================
# ENTRAÎNEMENT FINAL SUR TOUT LE DATASET + EXPORT DES MODÈLES
# ============================================================
# On ré-entraîne tous les modèles sur 100% des données d'entraînement
# avant de les exporter pour le backend.

print("Entraînement final du modèle FVC central sur tout le dataset...")
model_fvc.fit(X, y)

print("Entraînement final des 5 modèles quantile sur tout le dataset...")
for q, qmodel in quantile_models.items():
    print(f"  Entraînement quantile {q}...")
    qmodel.fit(X, y)

# Sauvegarde de tous les modèles au format JSON natif XGBoost
os.makedirs("../app/models", exist_ok=True)

# Modèle central
model_fvc.save_model("../app/models/model_fvc.json")
print("\n[OK] Modèle FVC central sauvegardé : backend/app/models/model_fvc.json")

# 5 modèles quantile
quantile_names = {
    0.025: "q005",
    0.10: "q020",
    0.50: "q050",
    0.90: "q080",
    0.975: "q095",
}  # IC 95% et IC 80%
for q, qmodel in quantile_models.items():
    filename = f"../app/models/model_{quantile_names[q]}.json"
    qmodel.save_model(filename)
    print(
        f"[OK] Modèle quantile {q} sauvegardé : backend/app/models/model_{quantile_names[q]}.json"
    )

print("\nTous les modèles ont été sauvegardés avec succès !")
print(f"Colonnes d'entraînement : {list(X.columns)}")
print(f"Nombre de features : {X.shape[1]}")

In [ ]:
from IPython.display import display
from sklearn.metrics import r2_score, mean_squared_error

print("--- Évaluation Locale sur le set de Test (avec Quantile Regression ML) ---")

TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
TEST_CACHE_FILE = os.path.join(DATA_DIR, "radiomics_features_test_cache.csv")

# 1. Chargement
df_test_local = pd.read_csv(TEST_CSV)
test_patients = df_test_local["Patient"].unique()
print(
    f"Chargement de {len(df_test_local)} lignes pour {len(test_patients)} patients de test."
)

# 2. Pipeline Radiomique (Extraction ou Chargement depuis le cache de test)
if os.path.exists(TEST_CACHE_FILE):
    print("Chargement des features de test depuis le cache...")
    df_test_features = pd.read_csv(TEST_CACHE_FILE)
else:
    print("Aucun cache trouvé pour le test. Extraction des features en cours...")
    test_features_dict = {}
    for p_id in tqdm(test_patients, desc="Extraction Radiomique Test"):
        try:
            slices = load_and_sort_scan(p_id, base_dir=TEST_IMG_DIR)
            vol = get_pixels_hu(slices)
            vol_res = resample_volume(vol, slices)
            mask = generate_lung_mask(vol_res)
            test_features_dict[p_id] = extract_radiomics_features(vol_res, mask)
        except Exception as e:
            print(f"Erreur sur {p_id}: {e}")
            test_features_dict[p_id] = [3000, -500, 200, 0.05]

    df_test_features = (
        pd.DataFrame.from_dict(
            test_features_dict,
            orient="index",
            columns=["Vol_Lung_cm3", "Mean_HU", "Std_HU", "Fibrosis_Ratio"],
        )
        .reset_index()
        .rename(columns={"index": "Patient"})
    )
    df_test_features.to_csv(TEST_CACHE_FILE, index=False)

# 3. Ingénierie des caractéristiques
df_test_merged = pd.merge(df_test_local, df_test_features, on="Patient", how="inner")
base_test = (
    df_test_merged.sort_values(["Patient", "Weeks"])
    .groupby("Patient")
    .first()
    .reset_index()
)
base_test = base_test[["Patient", "Weeks", "FVC", "Percent"]].rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
df_test_final = pd.merge(df_test_merged, base_test, on="Patient", how="inner")
df_test_final["Weeks_Delta"] = df_test_final["Weeks"] - df_test_final["Base_Week"]
df_test_final = pd.get_dummies(
    df_test_final, columns=["Sex", "SmokingStatus"], drop_first=True
)

X_train_cols = X.columns.tolist()
for col in X_train_cols:
    if col not in df_test_final.columns:
        df_test_final[col] = 0
X_test_eval = df_test_final[X_train_cols]
y_test_real = df_test_final["FVC"]
y_test_real_array = y_test_real.to_numpy()

# 4. Prédictions FVC centrale + quantiles ML
print("\nGénération des prédictions quantile ML sur le set de test...")
preds_fvc_test = model_fvc.predict(X_test_eval)

q_preds_test = {}
for q, qmodel in quantile_models.items():
    q_preds_test[q] = qmodel.predict(X_test_eval)

# Sigma ML = demi-largeur de l'intervalle [Q05, Q95]
sigma_ml_test = np.maximum(
    (q_preds_test[0.975] - q_preds_test[0.025]) / 2.0, 70
)  # IC 95%

# 5. Métriques
mae_test = np.mean(np.abs(y_test_real_array - preds_fvc_test))
rmse_test = np.sqrt(mean_squared_error(y_test_real_array, preds_fvc_test))
r2_test = r2_score(y_test_real_array, preds_fvc_test)
mape_test = (
    np.mean(np.abs((y_test_real_array - preds_fvc_test) / y_test_real_array)) * 100
)
score_test = laplace_log_likelihood(y_test_real.values, preds_fvc_test, sigma_ml_test)

# Coverage des intervalles quantile
cov_90_test = np.mean(
    (y_test_real_array >= q_preds_test[0.025])
    & (y_test_real_array <= q_preds_test[0.975])
)  # IC 95%
cov_60_test = np.mean(
    (y_test_real_array >= q_preds_test[0.10])
    & (y_test_real_array <= q_preds_test[0.90])
)  # IC 80%

print("\n--- Métriques de Régression sur le Test Local ---")
print(f"MAE  : {mae_test:.2f} mL")
print(f"RMSE : {rmse_test:.2f} mL")
print(f"R²   : {r2_test:.4f}")
print(f"MAPE : {mape_test:.2f} %")
print(f"Laplace Log Likelihood (Score Kaggle) : {score_test:.4f}")
print("\n--- Qualité des Intervalles Quantile ML ---")
print(f"Coverage IC 95% [Q2.5, Q97.5] : {cov_90_test * 100:.1f}% (cible >= 95%)")
print(f"Coverage IC 80% [Q10, Q90] : {cov_60_test * 100:.1f}% (cible >= 80%)")

# 6. Tableau de comparaison avec les 5 quantiles
df_test_comparison = pd.DataFrame(
    {
        "Patient": df_test_final["Patient"],
        "Semaine": df_test_final["Weeks"],
        "Vraie_FVC": y_test_real.values,
        "FVC_Prédite": np.round(preds_fvc_test).astype(int),
        "Q2.5 (IC95 bas)": np.round(q_preds_test[0.025]).astype(int),
        "Q10 (IC80 bas)": np.round(q_preds_test[0.10]).astype(int),
        "Q50 (médiane)": np.round(q_preds_test[0.50]).astype(int),
        "Q90 (IC80 haut)": np.round(q_preds_test[0.90]).astype(int),
        "Q97.5 (IC95 haut)": np.round(q_preds_test[0.975]).astype(int),
        "Erreur_Absolue": np.round(np.abs(y_test_real.values - preds_fvc_test)).astype(
            int
        ),
        "Dans_IC_95%": (
            (y_test_real.values >= q_preds_test[0.025])
            & (y_test_real.values <= q_preds_test[0.975])
        ),
    }
)
df_test_comparison = df_test_comparison.sort_values(["Patient", "Semaine"])
display(df_test_comparison.head(15))

In [ ]:
# ============================================================
# VISUALISATION : Bandes de Confiance Imbriquées (Quantile ML)
# ============================================================
# On affiche 3 niveaux de confiance superposés :
#   - Bande large   [Q05, Q95]  : Intervalle de confiance à 90%
#   - Bande moyenne [Q20, Q80]  : Intervalle de confiance à 60%
#   - Ligne centrale Q50        : Médiane prédite par le modèle quantile
#   - Ligne FVC centrale        : Prédiction du modèle FVC principal

nb_graphiques = 30
patients_to_plot = df_test_comparison["Patient"].unique()[:nb_graphiques]
num_plots = len(patients_to_plot)

cols = 3
rows = max(1, math.ceil(num_plots / cols))
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))

if num_plots == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i in range(rows * cols):
    ax = axes[i]
    if i < num_plots:
        p_id = patients_to_plot[i]
        pd_data = df_test_comparison[df_test_comparison["Patient"] == p_id].sort_values(
            "Semaine"
        )
        weeks = pd_data["Semaine"]

        # Bande IC 90% [Q05, Q95] — la plus large, la plus transparente
        ax.fill_between(
            weeks,
            pd_data["Q2.5 (IC95 bas)"],
            pd_data["Q97.5 (IC95 haut)"],
            color="royalblue",
            alpha=0.15,
            label="IC 95% [Q2.5-Q97.5]",
        )

        # Bande IC 60% [Q20, Q80] — plus resserrée
        ax.fill_between(
            weeks,
            pd_data["Q10 (IC80 bas)"],
            pd_data["Q90 (IC80 haut)"],
            color="royalblue",
            alpha=0.30,
            label="IC 80% [Q10-Q90]",
        )

        # Médiane quantile (Q50)
        ax.plot(
            weeks,
            pd_data["Q50 (médiane)"],
            color="steelblue",
            linewidth=1.5,
            linestyle="--",
            label="Médiane Q50 (ML)",
        )

        # Prédiction centrale du modèle FVC
        ax.plot(
            weeks,
            pd_data["FVC_Prédite"],
            color="darkorange",
            linewidth=2,
            marker="x",
            markersize=4,
            label="FVC Prédite",
        )

        # Vraie valeur FVC
        ax.plot(
            weeks,
            pd_data["Vraie_FVC"],
            color="green",
            linewidth=2,
            marker="o",
            markersize=5,
            label="Vraie FVC",
        )

        # Mise en forme
        in_ic = pd_data["Dans_IC_95%"].mean() * 100
        ax.set_title(f"Patient: {p_id} | Dans IC95%: {in_ic:.0f}%", fontsize=10)
        ax.set_xlabel("Semaine")
        ax.set_ylabel("FVC (mL)")
        ax.legend(fontsize=7, loc="best")
        ax.grid(True, alpha=0.3)
    else:
        ax.axis("off")

plt.suptitle(
    "Intervalles de Confiance IC 95% & IC 80% — Quantile Regression ML (XGBoost)",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ÉVALUATION QUANTILE ML SUR LE DATASET D'ENTRAÎNEMENT
# ============================================================
# On applique les 6 modèles (FVC central + 5 quantiles) sur X/y (train complet)
# pour visualiser la qualité des intervalles de confiance sur les données vues.

from sklearn.metrics import r2_score, mean_squared_error

print("=== Application des modèles ML sur le dataset d'entraînement ===\n")

# 1. Prédictions FVC centrale + 5 quantiles sur TOUT le train
preds_fvc_train = model_fvc.predict(X)

q_preds_train = {}
for q, qmodel in quantile_models.items():
    q_preds_train[q] = qmodel.predict(X)

# 2. Sigma ML = demi-largeur [Q05, Q95]
sigma_ml_train = np.maximum(
    (q_preds_train[0.975] - q_preds_train[0.025]) / 2.0, 70
)  # IC 95%

# 3. Métriques globales sur le train
y_arr = y.to_numpy()
mae_train = np.mean(np.abs(y_arr - preds_fvc_train))
rmse_train = np.sqrt(mean_squared_error(y_arr, preds_fvc_train))
r2_train = r2_score(y_arr, preds_fvc_train)
mape_train = np.mean(np.abs((y_arr - preds_fvc_train) / np.maximum(y_arr, 1))) * 100
score_train = laplace_log_likelihood(y_arr, preds_fvc_train, sigma_ml_train)

cov_90_train = np.mean(
    (y_arr >= q_preds_train[0.025]) & (y_arr <= q_preds_train[0.975])
)  # IC 95%
cov_60_train = np.mean(
    (y_arr >= q_preds_train[0.10]) & (y_arr <= q_preds_train[0.90])
)  # IC 80%

print("--- Métriques de Régression sur le Train ---")
print(f"MAE  : {mae_train:.2f} mL")
print(f"RMSE : {rmse_train:.2f} mL")
print(f"R²   : {r2_train:.4f}")
print(f"MAPE : {mape_train:.2f} %")
print(f"Laplace Log Likelihood : {score_train:.4f}")
print("\n--- Qualité des Intervalles Quantile ML (Train) ---")
print(f"Coverage IC 95% [Q2.5, Q97.5] : {cov_90_train * 100:.1f}% (cible >= 95%)")
print(f"Coverage IC 80% [Q10, Q90] : {cov_60_train * 100:.1f}% (cible >= 80%)")

# 4. Construction du DataFrame de comparaison sur le train
df_train_comparison = pd.DataFrame(
    {
        "Patient": df_final["Patient"].values,
        "Semaine": df_final["Weeks"].values,
        "Vraie_FVC": y_arr,
        "FVC_Prédite": np.round(preds_fvc_train).astype(int),
        "Q2.5 (IC95 bas)": np.round(q_preds_train[0.025]).astype(int),
        "Q10 (IC80 bas)": np.round(q_preds_train[0.10]).astype(int),
        "Q50 (médiane)": np.round(q_preds_train[0.50]).astype(int),
        "Q90 (IC80 haut)": np.round(q_preds_train[0.90]).astype(int),
        "Q97.5 (IC95 haut)": np.round(q_preds_train[0.975]).astype(int),
        "Erreur_Absolue": np.round(np.abs(y_arr - preds_fvc_train)).astype(int),
        "Dans_IC_95%": (
            (y_arr >= q_preds_train[0.025]) & (y_arr <= q_preds_train[0.975])
        ),
    }
)
df_train_comparison = df_train_comparison.sort_values(["Patient", "Semaine"])

print("\n--- Aperçu du tableau (15 premières lignes) ---")
display(df_train_comparison.head(15))

# ── 5. Distribution des erreurs absolues ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(
    df_train_comparison["Erreur_Absolue"],
    bins=50,
    color="steelblue",
    edgecolor="white",
    alpha=0.85,
)
axes[0].axvline(
    mae_train,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"MAE = {mae_train:.0f} mL",
)
axes[0].set_title("Distribution des Erreurs Absolues (Train)", fontsize=12)
axes[0].set_xlabel("Erreur Absolue (mL)")
axes[0].set_ylabel("Fréquence")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── 6. Vraie FVC vs FVC Prédite (scatter) ──────────────────────────────────────
fvc_min = min(y_arr.min(), preds_fvc_train.min())
fvc_max = max(y_arr.max(), preds_fvc_train.max())
axes[1].scatter(y_arr, preds_fvc_train, alpha=0.3, s=10, color="darkorange")
axes[1].plot(
    [fvc_min, fvc_max],
    [fvc_min, fvc_max],
    "k--",
    linewidth=1.5,
    label="Prédiction parfaite",
)
axes[1].set_title(f"Vraie FVC vs FVC Prédite\nR\u00b2 = {r2_train:.4f}", fontsize=12)
axes[1].set_xlabel("Vraie FVC (mL)")
axes[1].set_ylabel("FVC Prédite (mL)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ── 7. Largeur des intervalles par semaine (boxplot) ────────────────────────────
df_train_comparison["Largeur_IC95"] = (
    df_train_comparison["Q97.5 (IC95 haut)"] - df_train_comparison["Q2.5 (IC95 bas)"]
)
df_train_comparison["Largeur_IC80"] = (
    df_train_comparison["Q90 (IC80 haut)"] - df_train_comparison["Q10 (IC80 bas)"]
)

week_groups = (
    df_train_comparison.groupby("Semaine")["Largeur_IC95"].median().reset_index()
)
axes[2].plot(
    week_groups["Semaine"],
    week_groups["Largeur_IC95"],
    color="royalblue",
    linewidth=2,
    label="Largeur médiane IC 95% [Q2.5–Q97.5]",
)
week_groups80 = (
    df_train_comparison.groupby("Semaine")["Largeur_IC80"].median().reset_index()
)
axes[2].plot(
    week_groups80["Semaine"],
    week_groups80["Largeur_IC80"],
    color="steelblue",
    linewidth=2,
    linestyle="--",
    label="Largeur médiane IC 80% [Q10–Q90]",
)
axes[2].set_title("Largeur Médiane des IC par Semaine (Train)", fontsize=12)
axes[2].set_xlabel("Semaine")
axes[2].set_ylabel("Largeur de l'Intervalle (mL)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(
    "Analyse Globale — Prédictions Quantile ML (Dataset Entraînement)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# ── 8. Graphiques par patient (bandes imbriquées) ──────────────────────────────
nb_graphiques = 30
patients_train = df_train_comparison["Patient"].unique()[:nb_graphiques]
num_plots = len(patients_train)

cols = 3
rows = max(1, math.ceil(num_plots / cols))
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))

if num_plots == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i in range(rows * cols):
    ax = axes[i]
    if i < num_plots:
        p_id = patients_train[i]
        pd_data = df_train_comparison[
            df_train_comparison["Patient"] == p_id
        ].sort_values("Semaine")
        weeks_p = pd_data["Semaine"]

        # Bande IC 90% [Q05, Q95]
        ax.fill_between(
            weeks_p,
            pd_data["Q2.5 (IC95 bas)"],
            pd_data["Q97.5 (IC95 haut)"],
            color="royalblue",
            alpha=0.15,
            label="IC 95% [Q2.5–Q97.5]",
        )

        # Bande IC 60% [Q20, Q80]
        ax.fill_between(
            weeks_p,
            pd_data["Q10 (IC80 bas)"],
            pd_data["Q90 (IC80 haut)"],
            color="royalblue",
            alpha=0.30,
            label="IC 80% [Q10–Q90]",
        )

        # Médiane quantile (Q50)
        ax.plot(
            weeks_p,
            pd_data["Q50 (médiane)"],
            color="steelblue",
            linewidth=1.5,
            linestyle="--",
            label="Médiane Q50 (ML)",
        )

        # Prédiction FVC centrale
        ax.plot(
            weeks_p,
            pd_data["FVC_Prédite"],
            color="darkorange",
            linewidth=2,
            marker="x",
            markersize=4,
            label="FVC Prédite",
        )

        # Vraie valeur FVC
        ax.plot(
            weeks_p,
            pd_data["Vraie_FVC"],
            color="green",
            linewidth=2,
            marker="o",
            markersize=5,
            label="Vraie FVC",
        )

        in_ic = pd_data["Dans_IC_95%"].mean() * 100
        ax.set_title(f"Patient: {p_id} | Dans IC95%: {in_ic:.0f}%", fontsize=10)
        ax.set_xlabel("Semaine")
        ax.set_ylabel("FVC (mL)")
        ax.legend(fontsize=7, loc="best")
        ax.grid(True, alpha=0.3)
    else:
        ax.axis("off")

plt.suptitle(
    "Intervalles de Confiance IC 95% & IC 80% — Dataset Entraînement",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()
print(
    f"\nDone. {num_plots} patients affichés sur {len(df_train_comparison['Patient'].unique())} au total."
)